In [9]:
import sys, subprocess
subprocess.run(['nvidia-smi'], check=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','py-boost','cupy-cuda12x','pyarrow','scikit-learn','pandas'])
import cupy as cp
assert cp.cuda.runtime.getDeviceCount() > 0
print('CUDA:', cp.cuda.runtime.runtimeGetVersion())


CUDA: 12090


In [11]:
from pathlib import Path
# from google.colab import drive
# drive.mount('/content/drive')
DATA_DIR=Path('/content/drive/MyDrive/kaggle_single_cell')
OUT=Path('/content/pyboost_results'); OUT.mkdir(exist_ok=True)
SEED=42
ROUNDS=120
PCA_DIM=160
RUN_FINAL=True


In [ ]:
from pathlib import Path
from google.colab import files

DATA_DIR = Path('/content')

if not any((DATA_DIR / name).exists()
           for name in ['de_train.parquet', 'BDD-main.zip']):
    print('请选择之前的 BDD-main.zip，或官方 de_train.parquet')
    uploaded = files.upload()

    for name, content in uploaded.items():
        if name in ['BDD-main.zip', 'de_train.parquet']:
            (DATA_DIR / name).write_bytes(content)

assert any((DATA_DIR / name).exists()
           for name in ['de_train.parquet', 'BDD-main.zip']), \
    '请上传 BDD-main.zip 或 de_train.parquet，并保留原文件名'

print('数据文件夹：', DATA_DIR)

请选择之前的 BDD-main.zip，或官方 de_train.parquet


Saving BDD-main.zip to BDD-main.zip
数据文件夹： /content


In [12]:
from pathlib import Path
from zipfile import ZipFile
import shutil

DRIVE_DIR = Path(
    "/content/drive/MyDrive/kaggle_single_cell"
)

# 后续训练从这里读取 Multiome 数据
MULTIOME_DIR = Path("/content/multiome_data")
MULTIOME_DIR.mkdir(parents=True, exist_ok=True)

required = [
    "multiome_var_meta.csv",
    "multiome_train.parquet",
    "multiome_obs_meta.csv",
]

for name in required:
    destination = MULTIOME_DIR / name

    if destination.exists():
        print("已经就绪：", name)
        continue

    source = DRIVE_DIR / name
    archive_path = DRIVE_DIR / f"{name}.zip"

    if source.exists():
        print("正在复制：", name)
        shutil.copy2(source, destination)

    elif archive_path.exists():
        print("正在解压：", archive_path.name)

        with ZipFile(archive_path) as archive:
            matches = [
                item for item in archive.infolist()
                if not item.is_dir()
                and Path(item.filename).name == name
                and not item.filename.startswith("__MACOSX/")
            ]

            if len(matches) != 1:
                raise ValueError(
                    f"{archive_path.name} 内未找到唯一的 {name}"
                )

            temporary = destination.with_suffix(
                destination.suffix + ".tmp"
            )

            with archive.open(matches[0]) as src:
                with temporary.open("wb") as dst:
                    shutil.copyfileobj(
                        src, dst, length=16 * 1024 * 1024
                    )

            temporary.replace(destination)

    else:
        raise FileNotFoundError(
            f"找不到 {name} 或 {name}.zip"
        )

    print(
        "完成：", name,
        f"({destination.stat().st_size / 1024**2:.1f} MB)"
    )

print("\nMultiome 数据路径：", MULTIOME_DIR)

FileNotFoundError: 找不到 multiome_var_meta.csv 或 multiome_var_meta.csv.zip

In [14]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
data_dir = drive_root / "kaggle_single_cell"

print("Drive 是否可访问：", drive_root.exists())
print("数据文件夹是否存在：", data_dir.exists())

if data_dir.exists():
    print("\n文件夹中的内容：")
    for path in data_dir.iterdir():
        print(path.name)

    print("\n包含 sample_submission 的文件位置：")
    for path in data_dir.rglob("*sample_submission*"):
        print(path)

Drive 是否可访问： True
数据文件夹是否存在： True

文件夹中的内容：
pyboost_results
adata_obs_meta.csv
adata_excluded_ids.csv
multiome_obs_meta.csv
id_map.csv
adata_train.parquet
de_train.parquet
multiome_var_meta.csv
multiome_train.parquet
sample_submission.csv

包含 sample_submission 的文件位置：
/content/drive/MyDrive/kaggle_single_cell/sample_submission.csv


In [18]:
# ============================================================
# 1. Imports and paths
# ============================================================
from pathlib import Path
from zipfile import ZipFile
from time import perf_counter
from importlib.metadata import version
import gc
import json
import hashlib

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import cupy as cp

from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA
from py_boost import GradientBoosting

# Previously uploaded training files
DATA_DIR = Path("/content")

# Read Multiome files directly from Google Drive
MULTIOME_DIR = Path("/content/drive/MyDrive/kaggle_single_cell")

# Persistent output directory
DRIVE_DIR = Path(
    "/content/drive/MyDrive/kaggle_single_cell"
)
OUT = DRIVE_DIR / "pyboost_results"

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
PCA_DIM = 160

RUN_CV = True
RUN_FINAL = True

PARAMS = {
    "loss": "mse",
    "ntrees": 120,
    "lr": 0.06,
    "max_depth": 3,
    "min_data_in_leaf": 1,
    "subsample": 0.9,
    "colsample": 0.9,
    "lambda_l2": 1.0,
    "es": 0,
    "seed": SEED,
    "verbose": 10,
}

assert cp.cuda.runtime.getDeviceCount() > 0, \
    "没有可用的 NVIDIA GPU，请先切换到 GPU 运行时"

print("GPU free / total GB:")
print([
    round(value / 1024**3, 2)
    for value in cp.cuda.runtime.memGetInfo()
])


# ============================================================
# 2. Find files
# ============================================================
# Look in the existing local directory and the Drive data folder.
def find_file(name):
    candidates = [
        DATA_DIR / name,
        DRIVE_DIR / name,
    ]

    for path in candidates:
        if path.is_file():
            return path

    return None


# ============================================================
# 3. Load the 614 known cell–drug pairs
# ============================================================
de_path = find_file("de_train.parquet")

if de_path is not None:
    print("Loading:", de_path)
    df = pd.read_parquet(de_path)

else:
    zip_path = find_file("BDD-main.zip")

    if zip_path is not None:
        print("Loading training splits from:", zip_path)

        with ZipFile(zip_path) as archive:
            def read_split(suffix):
                ending = (
                    "all_cell_types_80_20_seed42_"
                    + suffix + ".parquet"
                )

                matches = [
                    name for name in archive.namelist()
                    if name.endswith(ending)
                    and not name.startswith("__MACOSX/")
                ]

                if len(matches) != 1:
                    raise ValueError(
                        f"ZIP 中没有找到唯一的 {ending}："
                        f"{matches}"
                    )

                with archive.open(matches[0]) as stream:
                    return pd.read_parquet(stream)

            train_part = read_split("train")
            validation_part = read_split("validation")

            assert (
                train_part.columns.tolist()
                == validation_part.columns.tolist()
            )

            df = pd.concat(
                [train_part, validation_part],
                ignore_index=True
            )

        del train_part, validation_part

    else:
        # Also support an already-extracted BDD directory.
        roots = [
            DATA_DIR / "BDD-main",
            DRIVE_DIR / "BDD-main",
        ]

        loaded = False

        for root in roots:
            split_dir = root / "splits"

            train_path = split_dir / (
                "all_cell_types_80_20_seed42_train.parquet"
            )
            validation_path = split_dir / (
                "all_cell_types_80_20_seed42_validation.parquet"
            )

            if train_path.exists() and validation_path.exists():
                df = pd.concat(
                    [
                        pd.read_parquet(train_path),
                        pd.read_parquet(validation_path),
                    ],
                    ignore_index=True
                )
                loaded = True
                print("Loaded extracted BDD splits:", root)
                break

        if not loaded:
            raise FileNotFoundError(
                "没有找到 de_train.parquet、BDD-main.zip "
                "或解压后的 BDD-main/splits。"
                "请将训练文件放在 /content 或 Drive 的 "
                "kaggle_single_cell 文件夹。"
            )

df = df.reset_index(drop=True)

pairs = ["cell_type", "sm_name"]

assert len(df) == 614, f"训练行数不是 614：{len(df)}"
assert not df.duplicated(pairs).any()
assert df[pairs].notna().all().all()

print("\nKnown pairs by cell type:")
print(df.groupby("cell_type").size())


# ============================================================
# 4. Load official test pairs and gene order
# ============================================================
id_path = find_file("id_map.csv")
sample_path = find_file("sample_submission.csv")
sample_zip = find_file("sample_submission.csv.zip")

if id_path is None:
    raise FileNotFoundError(
        "找不到 id_map.csv，请放在 /content 或 Drive 数据文件夹。"
    )

idmap = pd.read_csv(id_path).reset_index(drop=True)

if sample_path is not None:
    sample_header = pd.read_csv(
        sample_path, nrows=0
    ).columns.tolist()

elif sample_zip is not None:
    with ZipFile(sample_zip) as archive:
        matches = [
            name for name in archive.namelist()
            if Path(name).name == "sample_submission.csv"
            and not name.startswith("__MACOSX/")
        ]

        if len(matches) != 1:
            raise ValueError(
                "提交模板 ZIP 中需要包含唯一的 sample_submission.csv"
            )

        with archive.open(matches[0]) as stream:
            sample_header = pd.read_csv(
                stream, nrows=0
            ).columns.tolist()

else:
    raise FileNotFoundError(
        "找不到 sample_submission.csv 或其 ZIP。"
    )

assert sample_header[0] == "id"
genes = sample_header[1:]

assert len(genes) == 18211
assert len(set(genes)) == len(genes)
assert set(genes).issubset(df.columns)

assert {"id", *pairs}.issubset(idmap.columns)
assert idmap["id"].is_unique
assert not idmap.duplicated(pairs).any()
assert idmap[pairs].notna().all().all()

Y = df[genes].to_numpy(dtype=np.float32)

assert np.isfinite(Y).all()

print("\nTarget matrix:", Y.shape)
print("Test pairs:", len(idmap))


# ============================================================
# 5. Build baseline expression features
# ============================================================
# Input expression features are NOT reduced with PCA.
#
# First average within donor × cell_type, counting sparse zeros.
# Then average the donor profiles with equal donor weights.
#
# A new cache filename avoids loading older preprocessing output.
PROFILE_PATH = (
    OUT / "baseline_expression_donor_balanced_v2.csv"
)

if PROFILE_PATH.exists():
    profiles = pd.read_csv(
        PROFILE_PATH,
        index_col=0
    )

    print("\nLoaded cached baseline profiles:", profiles.shape)

else:
    required = [
        "multiome_obs_meta.csv",
        "multiome_var_meta.csv",
        "multiome_train.parquet",
    ]

    missing = [
        name for name in required
        if not (MULTIOME_DIR / name).is_file()
    ]

    if missing:
        raise FileNotFoundError(
            "解压目录中缺少：" + ", ".join(missing)
            + "\n当前目录：" + str(MULTIOME_DIR)
        )

    meta = pd.read_csv(
        MULTIOME_DIR / "multiome_obs_meta.csv"
    )
    var = pd.read_csv(
        MULTIOME_DIR / "multiome_var_meta.csv"
    )

    # Handle CSV index columns.
    if "obs_id" not in meta.columns:
        first = meta.columns[0]
        if first.startswith("Unnamed:"):
            meta.rename(
                columns={first: "obs_id"},
                inplace=True
            )

    if "location" not in var.columns:
        first = var.columns[0]
        if first.startswith("Unnamed:"):
            var.rename(
                columns={first: "location"},
                inplace=True
            )

    assert {
        "obs_id", "donor_id", "cell_type"
    }.issubset(meta.columns), meta.columns.tolist()

    assert {
        "location", "feature_type"
    }.issubset(var.columns), var.columns.tolist()

    assert meta[
        ["obs_id", "donor_id", "cell_type"]
    ].notna().all().all()

    meta["obs_id"] = meta["obs_id"].astype(str)

    assert not meta["obs_id"].duplicated().any()

    expr_genes = (
        var.loc[
            var["feature_type"].eq("Gene Expression"),
            "location"
        ]
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    assert expr_genes, \
        "没有找到 Gene Expression 特征，请检查 var 表。"

    gene_index = pd.Series(
        np.arange(len(expr_genes)),
        index=expr_genes
    )

    meta["group"] = (
        meta.groupby(
            ["donor_id", "cell_type"],
            sort=True
        ).ngroup()
    )

    group_meta = (
        meta[["group", "donor_id", "cell_type"]]
        .drop_duplicates()
        .sort_values("group")
        .reset_index(drop=True)
    )

    n_cells = (
        meta.groupby("group")
        .size()
        .reindex(group_meta["group"])
        .to_numpy()
    )

    obs_group = meta.set_index("obs_id")["group"]

    sums = np.zeros(
        (len(group_meta), len(expr_genes)),
        dtype=np.float64
    )

    parquet = pq.ParquetFile(
        MULTIOME_DIR / "multiome_train.parquet"
    )

    expected = {
        "obs_id", "location", "normalized_count"
    }

    assert expected.issubset(
        parquet.schema_arrow.names
    ), parquet.schema_arrow.names

    print("\nBuilding baseline expression profiles...")

    for batch_no, batch in enumerate(
        parquet.iter_batches(
            batch_size=500_000,
            columns=[
                "obs_id",
                "location",
                "normalized_count",
            ]
        )
    ):
        frame = batch.to_pandas()

        gene_ids = (
            frame["location"]
            .astype(str)
            .map(gene_index)
        )

        keep = gene_ids.notna()

        if keep.any():
            selected = frame.loc[keep]

            group_ids = (
                selected["obs_id"]
                .astype(str)
                .map(obs_group)
            )

            assert group_ids.notna().all(), \
                "存在无法匹配 metadata 的 RNA obs_id。"

            values = selected[
                "normalized_count"
            ].to_numpy(dtype=np.float64)

            assert np.isfinite(values).all()
            assert (values >= 0).all()

            np.add.at(
                sums,
                (
                    group_ids.to_numpy(dtype=int),
                    gene_ids.loc[keep].to_numpy(dtype=int),
                ),
                values
            )

        if batch_no % 20 == 0:
            print("Processed batch:", batch_no)

    donor_means = sums / n_cells[:, None]

    profiles = pd.DataFrame(
        donor_means,
        columns=expr_genes
    )
    profiles["cell_type"] = (
        group_meta["cell_type"].to_numpy()
    )

    profiles = (
        profiles.groupby("cell_type", sort=True)
        .mean()
    )

    assert np.isfinite(profiles.to_numpy()).all()

    profiles.to_csv(PROFILE_PATH)

    group_meta.assign(n_cells=n_cells).to_csv(
        OUT / "baseline_group_counts_v2.csv",
        index=False
    )

    np.savez_compressed(
        OUT / "baseline_donor_profiles_v2.npz",
        means=donor_means,
        genes=np.asarray(expr_genes, dtype=str)
    )

    del sums, donor_means, frame
    gc.collect()

    print("Saved:", PROFILE_PATH)

assert profiles.index.is_unique
assert profiles.columns.is_unique
assert set(df["cell_type"]).issubset(profiles.index)
assert set(idmap["cell_type"]).issubset(profiles.index)
assert np.isfinite(profiles.to_numpy()).all()

print("\nBaseline expression features:", profiles.shape)


# ============================================================
# 6. Build pseudo-target validation folds
# ============================================================
assert "control" in df.columns

control = (
    df["control"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "1.0"])
)

anchor_drugs = set(
    df.loc[
        df["cell_type"].isin(
            ["B cells", "Myeloid cells"]
        ) & ~control,
        "sm_name"
    ]
)

target_cells = [
    "NK cells",
    "T cells CD4+",
    "T cells CD8+",
    "T regulatory cells",
]

folds = []

for cell in target_cells:
    val_mask = (
        df["cell_type"].eq(cell)
        & ~df["sm_name"].isin(anchor_drugs)
        & ~control
    ).to_numpy()

    train_mask = ~val_mask

    assert val_mask.sum() > 0, \
        f"没有验证行：{cell}"

    folds.append(
        (cell, train_mask, val_mask)
    )

    print(
        cell,
        "| train:", int(train_mask.sum()),
        "| validation:", int(val_mask.sum())
    )


# ============================================================
# 7. Experiment identity and persistent outputs
# ============================================================
def array_hash(values):
    return hashlib.sha256(
        np.ascontiguousarray(values).tobytes()
    ).hexdigest()


signature = {
    "params": PARAMS,
    "pca_dim": PCA_DIM,
    "pyboost_version": version("py-boost"),
    "target_hash": array_hash(Y),
    "profile_hash": array_hash(
        profiles.to_numpy(dtype=np.float32)
    ),
    "profile_cells": profiles.index.tolist(),
    "profile_genes": profiles.columns.tolist(),
    "train_pairs": df[pairs].values.tolist(),
    "test_pairs": idmap[pairs].values.tolist(),
    "test_ids": idmap["id"].astype(str).tolist(),
    "genes": genes,
    "fold_validation_pairs": {
        cell: df.loc[va, pairs].values.tolist()
        for cell, tr, va in folds
    },
}

run_id = hashlib.sha256(
    json.dumps(
        signature,
        sort_keys=True,
        ensure_ascii=False
    ).encode()
).hexdigest()[:12]

RUN_DIR = OUT / f"run_{run_id}"
RUN_DIR.mkdir(exist_ok=True)

(RUN_DIR / "configuration.json").write_text(
    json.dumps(
        signature,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

versions = {}

for package in [
    "py-boost",
    "cupy-cuda12x",
    "numpy",
    "pandas",
    "scikit-learn",
    "pyarrow",
]:
    try:
        versions[package] = version(package)
    except Exception:
        versions[package] = "not found"

(RUN_DIR / "versions.json").write_text(
    json.dumps(versions, indent=2),
    encoding="utf-8"
)

print("\nResults directory:", RUN_DIR)


# ============================================================
# 8. Feature construction and training
# ============================================================
def make_features(
    train_frame,
    query_frame,
    use_expression
):
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32
    )

    x_train = encoder.fit_transform(
        train_frame[pairs].astype(str)
    )
    x_query = encoder.transform(
        query_frame[pairs].astype(str)
    )

    if use_expression:
        train_expression = profiles.loc[
            train_frame["cell_type"].to_numpy()
        ].to_numpy(dtype=np.float32)

        query_expression = profiles.loc[
            query_frame["cell_type"].to_numpy()
        ].to_numpy(dtype=np.float32)

        x_train = np.concatenate(
            [x_train, train_expression],
            axis=1
        )
        x_query = np.concatenate(
            [x_query, query_expression],
            axis=1
        )

    return (
        np.ascontiguousarray(x_train),
        np.ascontiguousarray(x_query)
    )


def mrrmse(y_true, y_pred):
    difference = (
        np.asarray(y_true, dtype=np.float64)
        - np.asarray(y_pred, dtype=np.float64)
    )

    return float(
        np.sqrt(
            np.mean(difference ** 2, axis=1)
        ).mean()
    )


def clear_gpu():
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()


def fit_predict(
    train_frame,
    query_frame,
    y_train,
    use_expression,
    use_pca
):
    model = None
    reducer = None

    try:
        x_train, x_query = make_features(
            train_frame,
            query_frame,
            use_expression
        )

        if use_pca:
            reducer = PCA(
                n_components=min(
                    PCA_DIM,
                    len(y_train) - 1,
                    y_train.shape[1]
                ),
                svd_solver="randomized",
                random_state=SEED
            )

            targets = reducer.fit_transform(
                y_train
            ).astype(np.float32)

        else:
            # No output PCA: all 18,211 targets retained.
            targets = np.ascontiguousarray(
                y_train,
                dtype=np.float32
            )

        print(
            "Input shape:", x_train.shape,
            "| Output training shape:", targets.shape
        )

        model = GradientBoosting(**PARAMS)

        # Fixed rounds. Validation labels are not passed to fit().
        model.fit(x_train, targets)

        prediction = model.predict(x_query)

        if isinstance(prediction, cp.ndarray):
            prediction = cp.asnumpy(prediction)

        prediction = np.asarray(
            prediction,
            dtype=np.float32
        )

        if reducer is not None:
            prediction = reducer.inverse_transform(
                prediction
            )

        assert prediction.shape == (
            len(query_frame), len(genes)
        )
        assert np.isfinite(prediction).all()

        return prediction

    finally:
        del model, reducer
        clear_gpu()


EXPERIMENTS = [
    ("identity_pca160", False, True),
    ("expression_pca160", True, True),
]


# ============================================================
# 9. Run four-fold validation for all four configurations
# ============================================================
if RUN_CV:
    records = []

    for name, use_expression, use_pca in EXPERIMENTS:
        for fold_no, (cell, tr, va) in enumerate(folds):
            print("\n" + "=" * 60)
            print(name, "| validation:", cell)

            score_path = RUN_DIR / (
                f"{name}_fold{fold_no}_score.json"
            )
            pred_path = RUN_DIR / (
                f"{name}_fold{fold_no}_pred.npy"
            )

            if score_path.exists() and pred_path.exists():
                record = json.loads(
                    score_path.read_text()
                )
                records.append(record)
                print("Skipping completed run:", record)
                continue

            start = perf_counter()

            prediction = fit_predict(
                train_frame=df.loc[tr],
                query_frame=df.loc[va],
                y_train=Y[tr],
                use_expression=use_expression,
                use_pca=use_pca
            )

            record = {
                "model": name,
                "cell_type": cell,
                "n_train": int(tr.sum()),
                "n_validation": int(va.sum()),
                "mrrmse": mrrmse(Y[va], prediction),
                "seconds": perf_counter() - start
            }

            np.save(pred_path, prediction)

            score_path.write_text(
                json.dumps(record, indent=2),
                encoding="utf-8"
            )

            records.append(record)

            pd.DataFrame(records).to_csv(
                RUN_DIR / "cv_scores.csv",
                index=False
            )

            print(record)

            del prediction
            clear_gpu()

    scores = pd.DataFrame(records)

    print("\nMRRMSE by cell type:")
    print(
        scores.pivot(
            index="cell_type",
            columns="model",
            values="mrrmse"
        ).to_string()
    )

    summary = (
        scores.groupby("model")
        .agg(
            mean_fold_mrrmse=("mrrmse", "mean"),
            total_seconds=("seconds", "sum")
        )
    )

    weighted = (
        scores.assign(
            weighted_error=(
                scores["mrrmse"]
                * scores["n_validation"]
            )
        )
        .groupby("model")
        .agg(
            weighted_error=("weighted_error", "sum"),
            n_validation=("n_validation", "sum")
        )
    )

    summary["pooled_row_mrrmse"] = (
        weighted["weighted_error"]
        / weighted["n_validation"]
    )

    summary.to_csv(
        RUN_DIR / "cv_summary.csv"
    )

    print("\nValidation summary:")
    print(summary.to_string())


# ============================================================
# 10. Full training and official submission files
# ============================================================
if RUN_FINAL:
    known_pairs = set(
        map(tuple, df[pairs].to_numpy())
    )
    test_pairs = set(
        map(tuple, idmap[pairs].to_numpy())
    )

    assert not known_pairs.intersection(test_pairs)
    assert set(idmap["sm_name"]).issubset(
        set(df["sm_name"])
    )

    for name, use_expression, use_pca in EXPERIMENTS:
        submission_path = RUN_DIR / (
            f"{name}_submission.csv"
        )

        if submission_path.exists():
            header = pd.read_csv(
                submission_path,
                nrows=0
            ).columns.tolist()

            saved_ids = pd.read_csv(
                submission_path,
                usecols=["id"]
            )["id"].to_numpy()

            assert header == ["id"] + genes
            assert np.array_equal(
                saved_ids,
                idmap["id"].to_numpy()
            )

            print("Already saved:", submission_path)
            continue

        print("\n" + "=" * 60)
        print("Full training:", name)

        prediction = fit_predict(
            train_frame=df,
            query_frame=idmap,
            y_train=Y,
            use_expression=use_expression,
            use_pca=use_pca
        )

        submission = pd.DataFrame(
            prediction,
            columns=genes
        )

        submission.insert(
            0, "id", idmap["id"].to_numpy()
        )

        assert submission.shape == (
            len(idmap), len(genes) + 1
        )

        temporary_path = submission_path.with_suffix(
            ".csv.tmp"
        )

        submission.to_csv(
            temporary_path,
            index=False,
            float_format="%.8g"
        )

        temporary_path.replace(submission_path)

        print(
            "Saved:", submission_path,
            "| shape:", submission.shape
        )

        del prediction, submission
        clear_gpu()

print("\nFinished. Results saved to:")
print(RUN_DIR)

GPU free / total GB:
[14.46, 14.56]
Loading: /content/de_train.parquet

Known pairs by cell type:
cell_type
B cells                17
Myeloid cells          17
NK cells              146
T cells CD4+          146
T cells CD8+          142
T regulatory cells    146
dtype: int64

Target matrix: (614, 18211)
Test pairs: 255

Building baseline expression profiles...
Processed batch: 0
Processed batch: 20
Processed batch: 40
Processed batch: 60
Processed batch: 80
Processed batch: 100
Processed batch: 120
Processed batch: 140
Processed batch: 160
Processed batch: 180
Processed batch: 200
Processed batch: 220
Processed batch: 240
Processed batch: 260
Processed batch: 280
Processed batch: 300
Processed batch: 320
Processed batch: 340
Processed batch: 360
Processed batch: 380
Processed batch: 400
Processed batch: 420
Saved: /content/drive/MyDrive/kaggle_single_cell/pyboost_results/baseline_expression_donor_balanced_v2.csv

Baseline expression features: (6, 22787)
NK cells | train: 485 | validat

INFO:py_boost.callbacks.callback:Stdout logging level is INFO.


[15:30:57] GDBT train starts. Max iter 120, early stopping rounds 0


INFO:py_boost.callbacks.callback:GDBT train starts. Max iter 120, early stopping rounds 0


[15:31:04] Iter 0; 


INFO:py_boost.callbacks.callback:Iter 0; 


... (291 outputs left out)

INFO:py_boost.callbacks.callback:Iter 110; 


[15:45:26] Iter 119; 


INFO:py_boost.callbacks.callback:Iter 119; 


Saved: /content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474/expression_pca160_submission.csv | shape: (255, 18212)

Finished. Results saved to:
/content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474


In [19]:
from pathlib import Path
import json
import pandas as pd

# 优先使用本次训练的结果目录
if "RUN_DIR" in globals() and Path(RUN_DIR).exists():
    result_dir = Path(RUN_DIR)
else:
    root = Path(
        "/content/drive/MyDrive/kaggle_single_cell/pyboost_results"
    )
    candidates = [
        p for p in root.glob("run_*")
        if p.is_dir()
        and list(p.glob("*_fold*_score.json"))
    ]
    if not candidates:
        raise FileNotFoundError("没有找到已完成的验证结果")

    result_dir = max(
        candidates,
        key=lambda p: max(
            f.stat().st_mtime
            for f in p.glob("*_fold*_score.json")
        )
    )

print("读取结果目录：", result_dir)

records = []
for path in sorted(result_dir.glob("*_fold*_score.json")):
    record = json.loads(path.read_text())
    records.append(record)

if not records:
    raise FileNotFoundError("该目录中没有已完成的验证结果")

scores = pd.DataFrame(records)

print("\n每种细胞的 MRRMSE（越低越好）：")
display(
    scores.pivot(
        index="cell_type",
        columns="model",
        values="mrrmse"
    ).round(6)
)

summary = scores.groupby("model").agg(
    completed_folds=("cell_type", "nunique"),
    mean_fold_mrrmse=("mrrmse", "mean"),
    total_seconds=("seconds", "sum"),
)

weighted = scores.assign(
    error_sum=scores.mrrmse * scores.n_validation
).groupby("model").agg(
    error_sum=("error_sum", "sum"),
    evaluated_pairs=("n_validation", "sum"),
)

summary["pooled_mrrmse"] = (
    weighted.error_sum / weighted.evaluated_pairs
)

# 未完成四折的均值不用于完整模型排名
summary["complete"] = summary.completed_folds.eq(4)

print("\n汇总结果：")
display(summary.round(6))

print("\n已经生成的官方提交文件：")
submissions = list(result_dir.glob("*_submission.csv"))
for path in submissions:
    print(path.name)
if not submissions:
    print("暂无。程序可能在验证阶段报错，还没进入全量训练。")

读取结果目录： /content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474

每种细胞的 MRRMSE（越低越好）：


model,expression_pca160,identity_pca160
cell_type,,
NK cells,1.245749,1.397970
T cells CD4+,1.017624,1.178725
T cells CD8+,0.825923,0.913261
T regulatory cells,1.046972,1.051645



汇总结果：


,completed_folds,mean_fold_mrrmse,total_seconds,pooled_mrrmse,complete
model,,,,,
expression_pca160,4,1.034067,634.835874,1.034877,True
identity_pca160,4,1.135400,43.329078,1.136265,True



已经生成的官方提交文件：
identity_pca160_submission.csv
expression_pca160_submission.csv


In [20]:
from pathlib import Path
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

roots = [
    Path("/content/drive/MyDrive/kaggle_single_cell/pyboost_results"),
    Path("/content/pyboost_results"),
]

for root in roots:
    print(f"\n检查目录：{root}")

    if not root.exists():
        print("目录不存在")
        continue

    found = []
    for pattern in [
        "*score*.json",
        "*scores*.csv",
        "*summary*.csv",
        "*submission.csv",
    ]:
        found.extend(root.rglob(pattern))

    for path in sorted(set(found)):
        print(path)

    if not found:
        print("目录存在，但没有找到分数或提交文件")


检查目录：/content/drive/MyDrive/kaggle_single_cell/pyboost_results
/content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474/cv_scores.csv
/content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474/cv_summary.csv
/content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474/expression_pca160_fold0_score.json
/content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474/expression_pca160_fold1_score.json
/content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474/expression_pca160_fold2_score.json
/content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474/expression_pca160_fold3_score.json
/content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474/expression_pca160_submission.csv
/content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474/identity_pca160_fold0_score.json
/content/drive/MyDrive/kaggle_single_cell/pyboost_results/run_e927a290c474/identity_pca160_fold1_scor

In [21]:
from pathlib import Path
import json
import pandas as pd

RUN_DIR = Path(
    "/content/drive/MyDrive/kaggle_single_cell/"
    "pyboost_results/run_e644aa1804b9"
)

records = [
    json.loads(path.read_text())
    for path in RUN_DIR.glob("*_fold*_score.json")
]

scores = pd.DataFrame(records)

print("每种验证细胞的 MRRMSE（越低越好）：")
display(
    scores.pivot(
        index="cell_type",
        columns="model",
        values="mrrmse"
    ).round(6)
)

print("四折平均 MRRMSE：")
display(
    scores.groupby("model").agg(
        completed_folds=("cell_type", "nunique"),
        mean_mrrmse=("mrrmse", "mean")
    ).round(6)
)

每种验证细胞的 MRRMSE（越低越好）：


KeyError: 'cell_type'